# Track 3 v2, stage 2: GRPO that has something to learn

The GRPO attempt in the submission moved nothing, and the reason is visible in the logs: with
`num_generations=2` on a policy that already answers 96% of the mix, most groups had zero reward
variance, and the T4 profile silently capped the run at 30 steps.

This kernel changes the setting, not the algorithm:

- **Data:** `tasks_train_mix2.jsonl` (312 rows) now carries 96 abstention rows. Those rows have
  no oracle actions, so a fabricated expense and a clarifying question both score 0 on the graded
  reward's name and parameter terms. `--abstain-reward` adds the only signal that separates them
  (1.0 for a lone `finish` whose answer contains "?", 0.0 otherwise).
- **Steps:** the 30-step cap is gone; `--t4-profile auto` still caps the per-turn rollout size,
  which is what actually caused the OOM risk.
- **LR:** 3e-6 instead of 1e-6, since the previous run was 30 steps at a rate tuned for long runs.

**Wall clock on T4 x2 (~6.5 h).** Train 4 h, then safety + hard-30 (55 min) before hard-90
(90 min), so the safety number lands even if the session runs long.

Report the delta either way. If GRPO improves abstention and leaves hard-30 flat, that is a real
RL result on the safety axis. If it moves nothing again, the honest line is "GRPO at this budget
does not beat the SFT warm start", with the reward-variance logs as the evidence.

In [ ]:
import os
from pathlib import Path

from kaggle_secrets import UserSecretsClient

try:
    token = UserSecretsClient().get_secret("HF_TOKEN")
except Exception as exc:
    raise SystemExit(
        "HF_TOKEN is not attached. CohereLabs/tiny-aya-global is gated, so the token is "
        "needed to load the model, not only to upload. Add-ons -> Secrets, then Save & Run All."
    ) from exc

os.environ["HF_TOKEN"] = token
# bash cells cannot see this Python env; hand them the token through a private tmp file
# instead of the kernel output log.
env = Path("/tmp/hf_env")
env.write_text(f'export HF_TOKEN="{token}"\n', encoding="utf-8")
env.chmod(0o600)
print("HF_TOKEN set (not printed)")

!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
%%bash
set -euo pipefail
source /tmp/hf_env
cd /kaggle/working
rm -rf lexsi-agentic-hinglish
git clone -q https://github.com/PRINCE2-AI/lexsi-agentic-hinglish.git
cd lexsi-agentic-hinglish
git log --oneline -1
python -m pip install -q -r requirements.txt
python data/build_hard.py
python data/build_hard90.py
python data/build_safety.py
python data/build_abstain.py
python data/build_mix.py --abstain --out data/tasks_train_mix2.jsonl
python scripts/verify_all.py

## Merge `sft3`

`run_baseline.py --adapter` needs a merged base, and merging once is cheaper than reloading the
adapter for four evals.

In [ ]:
import torch
from huggingface_hub import snapshot_download
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

BASE = "CohereLabs/tiny-aya-global"
repo = snapshot_download("primce/tiny-aya-hinglish-lora", allow_patterns=["sft3_adapter/*"],
                         local_dir="/kaggle/working/sft3_repo")
MERGED = "/kaggle/working/sft3_merged"
print("adapter repo:", repo)

tokenizer = AutoTokenizer.from_pretrained(BASE)
model = AutoModelForCausalLM.from_pretrained(BASE, dtype=torch.float16, device_map="auto")
model = PeftModel.from_pretrained(model, repo, subfolder="sft3_adapter")
merged = model.merge_and_unload()
merged.save_pretrained(MERGED, safe_serialization=True)
tokenizer.save_pretrained(MERGED)
del model, merged
torch.cuda.empty_cache()
print("merged ->", MERGED)

## GRPO, 40 steps

Rewards: graded (format + tool name + parameter names + values + answer), the no-duplicate-call
efficiency reward, and the abstention reward. `frac_reward_zero_std` in the log is the number to
watch: it was 1.0 on the first attempt, and anything below that means the setting change worked.

In [ ]:
%%bash
set -uo pipefail
source /tmp/hf_env
cd /kaggle/working/lexsi-agentic-hinglish
if [ ! -f /kaggle/working/sft3_merged/config.json ]; then echo "SFT3 MERGE MISSING"; exit 0; fi
time python scripts/train_grpo.py --model /kaggle/working/sft3_merged \
  --tasks data/tasks_train_mix2.jsonl --max-steps 40 --generations 2 --lr 3e-6 \
  --efficiency --out runs/grpo3 || echo "GRPO FAILED"

## Evaluate the GRPO adapter

In [ ]:
%%bash
cd /kaggle/working/lexsi-agentic-hinglish
set -uo pipefail
source /tmp/hf_env
if [ ! -d runs/grpo3 ]; then echo "GRPO ADAPTER MISSING"; exit 0; fi

CUDA_VISIBLE_DEVICES=0 python scripts/run_baseline.py --model /kaggle/working/sft3_merged \
  --adapter runs/grpo3 --tasks data/tasks_safety_eval.jsonl --safety --no-4bit \
  --out runs/safety_grpo3 || echo "GRPO SAFETY FAILED"

CUDA_VISIBLE_DEVICES=0 python scripts/run_baseline.py --model /kaggle/working/sft3_merged \
  --adapter runs/grpo3 --tasks data/tasks_hard_eval.jsonl --no-4bit \
  --out runs/grpo3_hard30 || echo "GRPO HARD30 FAILED"

In [ ]:
%%bash
cd /kaggle/working/lexsi-agentic-hinglish
set -uo pipefail
source /tmp/hf_env
if [ ! -d runs/grpo3 ]; then echo "GRPO ADAPTER MISSING"; exit 0; fi
CUDA_VISIBLE_DEVICES=0 python scripts/run_baseline.py --model /kaggle/working/sft3_merged \
  --adapter runs/grpo3 --tasks data/tasks_hard90_eval.jsonl --no-4bit \
  --out runs/grpo3_hard90 || echo "GRPO HARD90 FAILED"

## Collect and upload

In [ ]:
import json
import os
from pathlib import Path

from huggingface_hub import HfApi

ROOT = Path("/kaggle/working/lexsi-agentic-hinglish")
os.chdir(ROOT)

RUNS_REPO = "primce/lexsi-runs"
MODEL_REPO = "primce/tiny-aya-hinglish-lora"
RUNS = ['safety_grpo3', 'grpo3_hard30', 'grpo3_hard90']
KEYS = (
    "n", "success", "tac", "arr", "ter", "rad", "overcall", "lcf", "answer_ok",
    "strict_rate", "calls", "steps", "parse_failures", "seconds",
    "safe", "utility", "attack_ledger", "attack_answer", "unplanned_calls",
    "abstained", "asked", "invented_expense", "injections_fired",
)

for name in RUNS:
    path = ROOT / "runs" / name / "summary.json"
    if not path.exists():
        print(f"{name}: MISSING")
        continue
    summary = json.loads(path.read_text())
    print(name, {key: summary[key] for key in KEYS if key in summary})
    families = summary.get("families") or {}
    for family, row in families.items():
        print(f"   {family}: success={row.get('success')} safe={row.get('safe')}")

token = os.environ.get("HF_TOKEN")
if not token:
    print("no HF_TOKEN in this kernel, skipping upload")
else:
    api = HfApi(token=token)
    for name in RUNS:
        folder = ROOT / "runs" / name
        if folder.exists():
            api.upload_folder(repo_id=RUNS_REPO, repo_type="dataset",
                              folder_path=str(folder), path_in_repo=name)
            print(f"{name} -> {RUNS_REPO}")
    adapter = ROOT / "runs" / "grpo3"
    if (adapter / "adapter_config.json").exists():
        api.upload_folder(repo_id=MODEL_REPO, folder_path=str(adapter), path_in_repo="grpo3_adapter",
                          commit_message="grpo3: 40 steps on mix2, graded + efficiency + abstain rewards")
        print("grpo3_adapter ->", MODEL_REPO)

Paste the block back. The docs need `safety_grpo3.abstained`, `grpo3_hard30.success`,
`grpo3_hard90.success`, and the `frac_reward_zero_std` line from the training log.